In [ ]:
var_desired_probability = .6
var_conf_percent = str(var_desired_probability * 100) + "%"

var_thresholds = {}
var_gmm_results = {}

for window, variance in variance_dict.items():
    vars_reshaped = variance.reshape(-1, 1) # reshape the variances
    gmm = BayesianGaussianMixture(n_components=2, n_init=100, tol=0.001) # create GMM
    gmm.fit(vars_reshaped) # fit the GMM to the data
    
    # sort components
    means = gmm.means_.flatten()

    sigmas = np.sqrt(gmm.covariances_.flatten())

    order = np.argsort(means)
    means, sigmas, weights = means[order], sigmas[order], gmm.weights_[order]
    
    # compute posteriors over value range
    x = np.linspace(np.min(variance), np.max(variance), 10000)
    # probability distribution function
    pdfs = np.array([w * norm.pdf(x, m, s) for w, m, s in zip(weights, means, sigmas)])
    denom = np.sum(pdfs, axis=0)
    
    denom[denom == 0] = 1e-12
    posteriors = pdfs / denom
    
    # confidence threshold
    # inconsistent group is index 1
    if np.any(posteriors[1] >= var_desired_probability):
        inconsistent_cutoff = x[np.where(posteriors[1] >= var_desired_probability)[0][-1]]
    else:
        inconsistent_cutoff = np.nan


    var_thresholds[window] = {
        "inconsistent_cutoff": inconsistent_cutoff,
        "means": means,
    }
    var_gmm_results[window] = {"means": means, "sigmas": sigmas, "weights": weights}

# display thresholds
for w, t in var_thresholds.items():
    print(f"Window {w}:")
    print(f"  {var_conf_percent} Inconsistent cutoff: {t['inconsistent_cutoff']:.4f}")

# Display

In [ ]:
for window in window_sizes: 
    results = var_gmm_results[window]
    means = results["means"]
    sigmas = results["sigmas"]
    variances = variance_dict[window]

    # compute Gaussian PDFs
    x = np.linspace(np.nanmin(variances), np.nanmax(variances), 10000)
    pdfs = [norm.pdf(x, mu, sigma) for mu, sigma in zip(means, sigmas)]
    
    # extract confidence cutoffs
    t_inconsistent = var_thresholds[window]["inconsistent_cutoff"]

    # plot histogram
    plt.figure(figsize=(6, 4))
    plt.hist(variances, bins=50, density=True, alpha=0.4, color='gray', label='Variance Data')

    # plot Gaussians
    colors = ['blue','orange']
    labels = ['Consistent', 'Inconsistent']
    for pdf, color, label in zip(pdfs, colors, labels):
        plt.plot(x, pdf, color=color, lw=2, label=label)

    # shaded regions for high-confidence areas
    if not np.isnan(t_inconsistent):
        plt.axvspan(x.min(), t_inconsistent, color='blue', alpha=0.1)

    # plot confidence cutoff
    if not np.isnan(t_inconsistent):
        plt.axvline(t_inconsistent, color='red', linestyle='--', lw=2,
                    label=f'{var_conf_percent} Learning cutoff ({t_inconsistent:.2f})')

    
    plt.title(f"GMM ({var_conf_percent} Confidence Cutoffs) — Window {window}")
    plt.xlabel("Variance")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()